In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Generating synthetic data with noise

# physics params
g = 9.8
h0 = 1.0
v0 = 10.0

def true_solution(t):
    return h0 + v0 * t + 0.5 * g * (t**2)

t_min, t_max = 0.0, 2.0
N_data = 10
t_data = np.linspace(t_min, t_max, N_data)

np.random.seed(0)
noise_level = 0.7
h_data_exact = true_solution(t_data)
h_data_noisy = h_data_exact + np.random.randn(N_data)

t_data_tensor = torch.tensor(t_data, dtype=torch.float32).view(-1, 1)
h_data_tensor = torch.tensor(h_data_noisy, dtype=torch.float32).view(-1, 1)

In [ ]:
class PINN(nn.Module):
    def __init__(self, n_hidden=20):
        super(PINN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, n_hidden),
            nn.Tanh(),
            nn.Linear(n_hidden, n_hidden),
            nn.Tanh(),
            nn.Linear(n_hidden, 1)
        )
    
    def forward(self, t):
        return self.net(t)

model = PINN(n_hidden=20)

In [4]:
# automatic diff using autograd
def derivative(y, x):
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True)[0]

In [ ]:
def physics_loss(model, t):
    t.requires_grad_(True)
    h_pred = model(t)
    dh_dt_pred = derivative(h_pred, t)
    dh_dt_true = v0 - g*t
    
    ode_loss = torch.mean((dh_dt_pred - dh_dt_true)**2)
    return ode_loss

def initial_condition_loss(model):
    t0 = torch.zeros(1, 1, dtype=torch.float32, requires_grad=False)
    h0_pred = model(t0)
    return (h0_pred - h0).pow(2).mean()

def data_loss(model, t_data, h_data):
    h_pred = model(h_data)
    return torch.mean((h_pred - h_data)**2)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

lambda_data = 1.0
lambda_ode = 1.0
lambda_ic = 1.0

num_epochs = 2000
print_every = 200

In [ ]:
model.train()
for epoch in range(num_epochs):
    optimizer.zero_grad()
    
    l_data = data_loss(model, t_data_tensor, h_data_tensor)
    l_ode = physics_loss(model, t_data_tensor)
    l_ic = initial_condition_loss(model)
    
    loss = lambda_data * l_data + lambda_ode * l_ode + lambda_ic * l_ic
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) & print_every == 0:
        print(f'Epoch {epoch + 1}/{num_epochs}: Total Loss = {loss.item():.6f}, Data Loss = {l_data.item():.6f}, ODE Loss = {l_ode.item():.6f}, IC Loss = {l_ic.item():.6f}')